Settings Update

In [ ]:
pip install llama-index-postprocessor-nvidia-rerank

In [ ]:
# All installed packages
# ! pip install llama-index-postprocessor-nvidia-rerank
# ! pip install chromadb llama-index-vector-stores-chroma
# ! pip install llama-index-llms-nvidia
# ! pip install llama-index-embeddings-nvidia
# ! pip install flashrank
# ! pip install llama-index-postprocessor-flashrank-rerank

In [ ]:
# 6. Now query as normal - Standard Way
# Pass this template to the Query Engine
# This tells the engine: "When you ask the LLM, use this specific format."
# query_engine = index.as_query_engine(
#     text_qa_template=qa_template
# )


# 6. Now query as normal - More advanced way it will uses Reranker
# pip install llama-index-postprocessor-nvidia-rerank

# from llama_index.postprocessor.nvidia_rerank import NVIDIARerank
# # You can use any reranker model here, e.g., Cohere, BAAI, or Nvidia
# MODEL_RERANKER = os.getenv("MODEL_RERANKER")
# reranker = NVIDIARerank(model=MODEL_RERANKER, top_n=3)

# query_engine = index.as_query_engine(
#     text_qa_template=qa_template,
#     similarity_top_k=10,  # Retrieve more chunks initially
#     node_postprocessors=[reranker], # Rerank and select the top 3
# )

# query = "what is the person name in document"
# response = query_engine.query(query)
# print(response)

In [ ]:
import os
# 0. Settings Update
from llama_index.core import Settings

# ENV LOAD
HF_TOKEN = os.getenv("HF_TOKEN") 
HF_MODEL = os.getenv("HF_MODEL") 
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") 

# --- STEP 1: SETUP HUGGING FACE EMBEDDINGS ---
from llama_index.embeddings.nvidia import NVIDIAEmbedding
Settings.embed_model = NVIDIAEmbedding(
    model_name="nvidia/nv-embedqa-mistral-7b-v2"
)


# --- STEP 2: SETUP NVIDIA LLM MODEL ---
from llama_index.llms.nvidia import NVIDIA
import os

Settings.llm = NVIDIA(
    # model="mistralai/mistral-7b-instruct-v0.2",
    model="meta/llama-3.1-8b-instruct",
    max_tokens=1024, # Keep this size reasonable
    temperature=0.1,
    stop_sequences=["\n\n", "</s>", "<|eot_id|>", "Answer:", "Thank you", "Here is"],
)

from llama_index.core import SimpleDirectoryReader

# 1. Loading data
base_folder = "./docs"
reader = SimpleDirectoryReader(base_folder, recursive=True, exclude_hidden=False)
docs = reader.load_data()

# 2. Creating Chunks --> Nodes
from llama_index.core.node_parser import SentenceSplitter
# node_parser = SentenceSplitter(chunk_size=500, chunk_overlap=50)
node_parser = SentenceSplitter(chunk_size=300, chunk_overlap=30)
nodes = node_parser.get_nodes_from_documents(docs, show_progress=True)

# 3. Indexing
from llama_index.core import VectorStoreIndex
index = VectorStoreIndex(nodes, )

# # 4. Persist index -> Storing Index
# index_dir = "./storage"
# index.storage_context.persist(index_dir)

# 4. Vector storage

from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb
from llama_index.core import StorageContext

chroma_db_path = "./chroma_db"
collection_name = "my_rag_collection"

# 1. Initialize a Chroma Client (in-memory or persistent)
chroma_client = chromadb.PersistentClient(path=chroma_db_path) # Persist to disk chroma_db_path
chroma_collection = chroma_client.get_or_create_collection(collection_name)

# 2. Create the ChromaVectorStore
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# 3. Use a custom StorageContext to inject the vector store
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 4. Create the index using the storage_context
index = VectorStoreIndex(nodes, storage_context=storage_context)

# Note: The index.storage_context.persist() step is then managed by Chroma's persistence

# 5. Querying the Data
from llama_index.core import PromptTemplate

# Define your System Prompt and the Template Structure
# Zephyr uses specific tags: <|system|>, <|user|>, <|assistant|>
# We embed your "System Prompt" at the top.
qa_template_str = (
    "<|system|>\n"
    "You are a strict document assistant. "
    "Answer strictly based on the context provided below. "
    "If the answer is not in the context, say 'I do not know'.\n"
    "</s>\n"
    "<|user|>\n"
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, "
    "answer the query.\n"
    "Query: {query_str}\n"
    "</s>\n"
    "<|assistant|>\n"
)

# Create the Prompt Template Object
qa_template = PromptTemplate(qa_template_str)


from llama_index.postprocessor.flashrank_rerank import FlashRankRerank

# MODEL_RERANKER = os.getenv("MODEL_RERANKER")
MODEL_RERANKER = "nvidia/nv-rerankqa-mistral-4b-v3"
# reranker = NVIDIARerank(model=MODEL_RERANKER, top_n=3)

# --- STEP 1: DEFINE THE LOCAL RERANKER ---
local_reranker = FlashRankRerank(
    # The model will download on the first run, but subsequent runs are local and fast.
    model_name="bge-reranker-base", 
    top_n=3
)

from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.chat_engine import ContextChatEngine

# Your RAG instruction and persona
SYSTEM_PROMPT = (
    "You are a strict document assistant. "
    "Answer strictly based on the context provided. "
    "If the answer is not in the context, say 'I do not know'. "
    "Always maintain a concise and factual tone."
)

# 1. Create a custom retriever that incorporates the Reranker
retriever = index.as_retriever(similarity_top_k=5)

# 2. Create the Chat Engine
chat_engine = ContextChatEngine.from_defaults(
    retriever=retriever,
    node_postprocessors=[local_reranker], # Your fast Flashrank reranker
    memory=ChatMemoryBuffer.from_defaults(token_limit=3000),
    
    # --- Inject your custom instructions here ---
    system_prompt=SYSTEM_PROMPT,
)

# Use the streaming method:
streaming_response = chat_engine.stream_chat("What is the person's name?")

print("\n[Streaming Response]:")
streaming_response.print_response_stream()

In [ ]:

# query_engine = index.as_query_engine(
#     text_qa_template=qa_template,
#     similarity_top_k=3,  # Retrieve more chunks initially
#     node_postprocessors=[local_reranker], # Rerank and select the top 3
#     streaming=True
# )

# # Perform a streaming query
# query = "What are his skills sets mainly?"
# streaming_response = query_engine.query(query)

# print("\n[Streaming Response]:")
# streaming_response.print_response_stream()

2025-11-21 00:41:09,194 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/embeddings "HTTP/1.1 200 OK"



[Streaming Response]:


2025-11-21 00:41:09,954 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"


Prakasha Bhajantri.

In [18]:
# Use the streaming method:
streaming_response = chat_engine.stream_chat("What he is responsibilities in IBM?")

print("\n[Streaming Response]:")
streaming_response.print_response_stream()

2025-11-21 00:41:52,794 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/embeddings "HTTP/1.1 200 OK"



[Streaming Response]:


2025-11-21 00:41:53,455 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"


Senior Data Scientist.

In [19]:
# Use the streaming method:
streaming_response = chat_engine.stream_chat("What all works he has done while working with IBM?")

print("\n[Streaming Response]:")
streaming_response.print_response_stream()

2025-11-21 00:42:30,134 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/embeddings "HTTP/1.1 200 OK"



[Streaming Response]:


2025-11-21 00:42:30,775 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"


According to the context, the person has worked on the following projects while at IBM:

1. Cost forecasting model
2. GenAI Assistants (Interactive Chatbots)
3. A/B Testing
4. Pokémon Game Analytics Engine
5. Object Detection
6. Credit Risk Modeling
7. Customer Segmentation

Additionally, he has also led the development of a GenAI LLM-based RCA Assistant for Network Operation Centre, prepared a dataset and deployed a YOLO-based object detection engine, and designed and implemented a GenAI-powered RAG based Meeting Summarizer Assistant integrated with ServiceNow.

In [ ]:
# # Perform a streaming query
# query = "Which location he is located?"
# streaming_response = query_engine.query(query)

# print("\n[Streaming Response]:")
# streaming_response.print_response_stream()

# # Perform a streaming query
# query = "is he open to relocation any countries?"
# streaming_response = query_engine.query(query)

# print("\n[Streaming Response]:")
# streaming_response.print_response_stream()



####### END

In [ ]:
from llama_index.core import SimpleDirectoryReader

base_folder = "./docs"
reader = SimpleDirectoryReader(base_folder, recursive=True, exclude_hidden=False)

In [ ]:
reader.input_files

In [ ]:
docs = reader.load_data()
docs

In [ ]:
len(docs)

In [ ]:
print(docs[0].get_metadata_str())

In [ ]:
docs[0].__dict__

In [ ]:
from llama_index.core.node_parser import SentenceSplitter
node_parser = SentenceSplitter(chunk_size=500, chunk_overlap=0)

In [ ]:
nodes = node_parser.get_nodes_from_documents(docs, show_progress=True)
len(nodes)

In [ ]:
nodes

### Indexing

In [ ]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
# pip install llama-index-embeddings-huggingface

# --- STEP 1: SETUP HUGGING FACE EMBEDDINGS ---
# This tells LlamaIndex to use the local BAAI model instead of OpenAI
# (Make sure you run this BEFORE creating the index)
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

# --- STEP 2: CREATE YOUR INDEX ---
# Now, when this runs, it will look at Settings, see the HF model, 
# and use your CPU to create embeddings without asking for an OpenAI key.

In [ ]:
from llama_index.core import VectorStoreIndex

In [ ]:
index = VectorStoreIndex(nodes, )

In [ ]:
# Persist index

index_dir = "./storage"
index.storage_context.persist(index_dir)

In [ ]:
# Load index
from llama_index.core import load_index_from_storage, StorageContext

# rebuild storage contenxt
storage_context = StorageContext.from_defaults(persist_dir=index_dir)

# load index
index = load_index_from_storage(storage_context)

In [ ]:
# !pip install llama-index-llms-huggingface-api

In [ ]:
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
# from llama_index.llms.huggingface import HuggingFaceLLM
import os
# 1. Get your Hugging Face Token (from https://huggingface.co/settings/tokens)
# It's best to check if it exists to avoid hidden errors
HF_TOKEN = os.getenv("HF_TOKEN") 
HF_MODEL = os.getenv("HF_MODEL") 


# 2. Configure the LLM (The "Brain" that writes the answer)
# We use the Inference API so you don't need a massive GPU on your laptop.
Settings.llm = HuggingFaceInferenceAPI(
    model_name=HF_MODEL,
    # model_name = "Qwen/Qwen2.5-7B-Instruct",
    token=HF_TOKEN
)

In [ ]:
from llama_index.core import PromptTemplate

# 1. Define your System Prompt and the Template Structure
# Zephyr uses specific tags: <|system|>, <|user|>, <|assistant|>
# We embed your "System Prompt" at the top.
qa_template_str = (
    "<|system|>\n"
    "You are a strict document assistant. "
    "Answer strictly based on the context provided below. "
    "If the answer is not in the context, say 'I do not know'.\n"
    "</s>\n"
    "<|user|>\n"
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, "
    "answer the query.\n"
    "Query: {query_str}\n"
    "</s>\n"
    "<|assistant|>\n"
)

# 2. Create the Prompt Template Object
qa_template = PromptTemplate(qa_template_str)

# 3. Pass this template to the Query Engine
# This tells the engine: "When you ask the LLM, use this specific format."
query_engine = index.as_query_engine(
    text_qa_template=qa_template
)

In [ ]:
# 4. Now query as normal
query = "what is the person name in document"
response = query_engine.query(query)
print(response)

In [ ]:
retriever = index.as_retriever()

In [ ]:
input = "what is the person name in document"
retriever.retrieve(input)

In [ ]:
## Chat engine
chat = index.as_chat_engine(verbose=True)


In [ ]:
chat.chat("hello")

In [ ]:
chat.chat("Who's resume is this?")

In [ ]:
chat.chat("Which company he is working currently")

In [ ]:
#### Loading existing index to create chat engine

In [ ]:
def wrapper_chat_history(memory):
    chat_history = []
    for m in memory:
        if m.role in ['user', 'assistant'] and m.content is not None:
            chat_history.append(m.content)
    return chat_history

In [ ]:
wrapper_chat_history(chat.chat_history)

In [ ]:
def converse(message, chat_history):
    response = chat.chat(message)
    chat_history = wrapper_chat_history(chat.chat_history)
    return response.response

In [ ]:
chat.reset()

In [ ]:
import gradio as gr

try:
    demo.close()

except:
    pass

demo = gr.ChatInterface(fn=converse)

demo.launch(share=False)

In [ ]:
# !pip install gradio